# CNN-GAF Cloudburst Prediction — Model Evaluation

**Paper:** Precision forecasting of cloudbursts with CNNs and GAF for real-time disaster response  
**DOI:** [10.1007/s40808-026-02826-4](https://doi.org/10.1007/s40808-026-02826-4)

Evaluates the pre-trained 6-hour and 12-hour models on training and hold-out test data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from keras.models import load_model
from sklearn.metrics import (
    confusion_matrix, precision_recall_fscore_support,
    roc_curve, auc, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
# Load 6-hour dataset  (change to CB12/NB for 12H evaluation)
CB_DATA   = 'CB6.npy'
NB_DATA   = 'NB.npy'
TEST_DATA = 'TEST6.npy'

cloudburst_data     = np.load(CB_DATA)
non_cloudburst_data = np.load(NB_DATA)

X = np.concatenate((cloudburst_data, non_cloudburst_data))
y = np.concatenate((np.ones(cloudburst_data.shape[0]),
                    np.zeros(non_cloudburst_data.shape[0])))

rng = np.random.default_rng(seed=42)
idx = rng.permutation(X.shape[0])
X, y = X[idx], y[idx]

print(f"Dataset: X={X.shape}  y={y.shape}")
print(f"Cloudburst: {int(y.sum())}  Non-cloudburst: {int((1-y).sum())}")

## 2. GAF Image Visualisation

In [ ]:
FEATURE_LABELS = [
    "Rainfall (mm)", "Temperature (°C)", "Relative Humidity (%)",
    "Wind Speed 10m (Kt)", "SLP (hPa)", "MSLP (hPa)"
]
font = fm.FontProperties(family='DejaVu Sans', size=9)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
fig.suptitle("GAF Images — Sample 0 (Cloudburst)", fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    im = ax.imshow(X[0][i], cmap='viridis')
    ax.set_title(FEATURE_LABELS[i], fontproperties=font)
    ax.axis('off')
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 2, figsize=(8, 11))
fig.suptitle("GAF Images — Sample 1", fontsize=13, fontweight='bold')
for i in range(3):
    for j in range(2):
        idx_f = 2 * i + j
        axes[i, j].imshow(X[1][idx_f], cmap='plasma')
        axes[i, j].set_title(FEATURE_LABELS[idx_f], fontproperties=font)
        axes[i, j].axis('off')
plt.tight_layout()
plt.show()

## 3. Load Pre-trained Model

In [ ]:
model = load_model('Model6.h5')
model.summary()

## 4. Model Evaluation

In [ ]:
y_pred_prob = model.predict(X, verbose=0).flatten()
y_pred      = np.round(y_pred_prob).astype(int)
y_true      = y.astype(int)

cm = confusion_matrix(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
accuracy = (cm[0,0] + cm[1,1]) / cm.sum()

print("=" * 40)
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 40)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Non-CB', 'Cloudburst'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Model6 (6H)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2,
         label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.02])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Model6 (6-Hour Prediction)', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"ROC-AUC: {roc_auc:.4f}")

## 6. Hold-out Test Set Predictions

> **Note:** The 3rd event in TEST6 / TEST12 is a repeated event from the training set — its result should be ignored.

In [ ]:
test = np.load(TEST_DATA)
print(f"Test set shape: {test.shape}")

predictions = model.predict(test, verbose=0)
probabilities = (predictions * 100).flatten()

print("\nCloudburst Probability per test event:")
print("-" * 35)
for i, p in enumerate(probabilities):
    flag = " [IGNORE - repeated event]" if i == 2 else ""
    label = "HIGH" if p >= 75 else ("MODERATE" if p >= 40 else "LOW")
    print(f"  Event {i+1}: {p:6.2f}%  [{label}]{flag}")